In [40]:
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd

def get_season_stats(season='2023-24'):
    # Fetching season stats for all players
    season_stats = leaguedashplayerstats.LeagueDashPlayerStats(season=season)
    season_stats_df = season_stats.get_data_frames()[0]
    return season_stats_df

# Get the DataFrame of season stats for all players in the 2023-24 season
season_stats_df = get_season_stats()

# Display all columns
pd.set_option('display.max_columns', None)

In [41]:
# Save the player names before dropping the column
player_names = player_names.reset_index(drop=True)

# Columns to drop
columns_to_drop = ['PLAYER_ID','PLAYER_NAME','NICKNAME','TEAM_ID','TEAM_ABBREVIATION','W','L','W_PCT','NBA_FANTASY_PTS','DD2','TD3','WNBA_FANTASY_PTS','GP_RANK','W_RANK','L_RANK','W_PCT_RANK','MIN_RANK','FGM_RANK','FGA_RANK','FG_PCT_RANK','FG3M_RANK','FG3A_RANK','FG3_PCT_RANK','FTM_RANK','FTA_RANK','FT_PCT_RANK','OREB_RANK','DREB_RANK','REB_RANK','AST_RANK','TOV_RANK','STL_RANK','BLK_RANK','BLKA_RANK','PF_RANK','PFD_RANK','PTS_RANK','PLUS_MINUS_RANK','NBA_FANTASY_PTS_RANK','DD2_RANK','TD3_RANK','WNBA_FANTASY_PTS_RANK']

# Filter out columns that exist in the DataFrame
columns_to_drop = [col for col in columns_to_drop if col in season_stats_df.columns]

# Drop the filtered columns
numerical_df = season_stats_df.drop(columns_to_drop, axis=1)

# Check the resulting DataFrame
print(numerical_df.head())


    AGE  GP          MIN  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  FTM  FTA  \
0  23.0  19   189.226667   31   66   0.470    11    33    0.333   11   15   
1  24.0  26   204.463333   30   70   0.429    25    60    0.417    4    4   
2  20.0  15   125.430000   12   40   0.300     9    30    0.300    2    2   
3  28.0  37  1189.035000  204  374   0.545    20    70    0.286   81  130   
4  27.0  38   634.948333   87  202   0.431    44   111    0.396   23   26   

   FT_PCT  OREB  DREB  REB  AST  TOV  STL  BLK  BLKA  PF  PFD  PTS  PLUS_MINUS  
0   0.733     9    16   25   10    9    9    2     6  14    9   84          47  
1   1.000     5    16   21   16    0    0    1     0  23    4   89          26  
2   1.000     2    11   13    4    6    1    1     2   6    1   35         -28  
3   0.623   100   148  248  118   51   30   20    26  66  117  509         218  
4   0.885    10    60   70   60   27   18    3    10  55   29  241          43  


In [42]:
from scipy.spatial.distance import pdist, squareform
import numpy as np

# Select the columns representing the attributes we care about
attributes = numerical_df [['AGE', 'GP', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS']]
# Calculate the pairwise distances
distances = pdist(attributes.values, metric='euclidean')

# Convert the distances to a square matrix
dist_matrix = squareform(distances)

In [43]:
# Make a copy of the distance matrix
dist_matrix_copy = np.copy(dist_matrix)

# Set the diagonal to infinity so that we ignore the zero values
np.fill_diagonal(dist_matrix, np.inf)

# Find the indices of the minimum value
min_indices = np.unravel_index(np.argmin(dist_matrix), dist_matrix.shape)

print("The most similar players are at indices:", min_indices)

The most similar players are at indices: (161, 318)


In [44]:
player1 = season_stats_df.iloc[242]
player2 = season_stats_df.iloc[414]

print("The most similar players are:", player1['PLAYER_NAME'], "and", player2['PLAYER_NAME'])

The most similar players are: Javonte Smart and Omer Yurtseven


In [45]:
import plotly.graph_objects as go

def find_most_similar(player_name, df, dist_matrix):
    # Get the index of the selected player
    player_index = df[df['PLAYER_NAME'] == player_name].index[0]

    # Set the diagonal to infinity to ignore self-comparison
    np.fill_diagonal(dist_matrix, np.inf)

    # Find the index of the most similar player
    similar_player_index = np.argmin(dist_matrix[player_index])

    return df.iloc[[player_index, similar_player_index]]

In [46]:
def create_plotly_table(df):
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(df.columns),
                    fill_color='paleturquoise',
                    align='left'),
        cells=dict(values=[df[k].tolist() for k in df.columns],
                   fill_color='lavender',
                   align='left'))
    ])

    fig.update_layout(width=500, height=300)
    return fig

In [47]:
# Find the most similar players
min_indices = np.unravel_index(np.argmin(dist_matrix), dist_matrix.shape)
most_similar_player_names = player_names.iloc[most_similar_players_indices]

# Map indices back to player names
most_similar_player_names = player_names.iloc[most_similar_players_indices]

# Now you can print or use the player names along with the stats
print("The most similar players are:", most_similar_player_names.iloc[0], "and", most_similar_player_names.iloc[1])


The most similar players are: Frank Ntilikina and Kevon Harris


In [48]:
# Create a function to find and display the most similar player
def find_and_display_similar(player_name):
    player_index = player_names[player_names == player_name].index[0]
    similar_player_index = np.argmin(dist_matrix[player_index])
    most_similar_players = numerical_df.iloc[[player_index, similar_player_index]]
    
    # Extract relevant columns for the table and transpose
    table_df = most_similar_players[['AGE', 'GP', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT']].T
    table_df.columns = player_names.iloc[[player_index, similar_player_index]].values
    
    # Create and show the Plotly table with transposed DataFrame
    fig = go.Figure(data=[go.Table(
        header=dict(values=['Metric'] + table_df.columns.tolist(),
                    fill_color='paleturquoise',
                    align='left'),
        cells=dict(values=[table_df.index.tolist()] + [table_df[col].tolist() for col in table_df.columns],
                   fill_color='lavender',
                   align='left'))
    ])
    
    # Update layout for a better fit
    fig.update_layout(width=700, height=500)
    fig.show()


# Create a dropdown widget for player names
player_dropdown = Dropdown(options=player_names.tolist())

# Use 'interact' to create the interactivity
interact(find_and_display_similar, player_name=player_dropdown)

interactive(children=(Dropdown(description='player_name', options=('A.J. Lawson', 'AJ Green', 'AJ Griffin', 'A…

<function __main__.find_and_display_similar(player_name)>